# El modelo PHREEQC

**PHREEQC** puede realizar reacciones por lotes (*batch reactions*), que consisten en mezclar diferentes tipos de agua, o agua con minerales, gases, etc., para crear un sistema (el “balde”). El modelo calcula la distribución de especies acuosas, gaseosas y sólidas dentro del sistema especificado. PHREEQC incluye capacidades para modelar **adsorción**, **cinética** y **transporte** de especies acuosas dentro del sistema.

Puedes acceder al código de PHREEQC y encontrar soporte en línea en:  
[http://wwwbrr.cr.usgs.gov/projects/GWC_coupled/phreeqc/index.html](http://wwwbrr.cr.usgs.gov/projects/GWC_coupled/phreeqc/index.html)

La versión más reciente de PHREEQC es la **3.7.3**, que es la utilizada en este curso. Otra alternativa útil es la versión en lote (*batch*), que permite ejecutar trabajos grandes o enlazar PHREEQC con otros programas. Los usuarios también pueden descargar **PhreeqcI**, que tiene una interfaz interactiva para ingresar datos. Todas las versiones usan el mismo código base y resuelven los problemas de la misma forma.

## Tipos de comandos en PHREEQC

El modelo PHREEQC utiliza **tres tipos principales de palabras clave (comandos)** que se ingresan mediante archivos de texto:

1. **Comandos de gestión (manager)**:
   - `KNOBS_PRINT`, `TITLE`, `USE`, `END`, `SAVE`, `SELECTED_OUTPUT`, `SOLUTION_SPREAD`, entre otros.

2. **Comandos para definir el sistema**:
   - `SOLUTION_SPECIES`, `MASTER`, `SOLUTION`, `SYSTEM_SPECIES`, `SURFACE_SPECIES`, `GAS_PHASE`, `SOLID_SOLUTIONS`, `RATES`, etc.

3. **Comandos de acción por lote o transporte**:
   - `SOLUTION`, `REACTION`, `INCREMENTAL_REACTIONS`, `MIX`, `SURFACE`, `ADVECTION`, `INVERSE_MODELING`, `EXCHANGE`, `EQUILIBRIUM_PHASES`, `KINETICS`, entre otros.

Los bloques de comandos pueden combinarse en un único archivo de entrada para simular situaciones complejas. Las reacciones por lote se definen e implementan automáticamente siempre que una solución (`SOLUTION`) se combine con algún otro bloque clave como `EXCHANGE`, `EQUILIBRIUM_PHASES`, `GAS_PHASE`, `KINETICS`, `MIX`, `REACTION`, `REACTION_TEMPERATURE`, `SOLID_SOLUTIONS` o `SURFACE`, **y no estén separados por un `END`**.

Cada comando puede tener **subcomandos** que refinan los cálculos. Por ejemplo, el comando `SOLUTION` puede incluir subcomandos como `temp`, `units`, `pe` y `pH` para establecer esas condiciones.

Es posible importar datos desde hojas de cálculo y exportar resultados seleccionados a archivos de texto para análisis posteriores. Los datos enviados al archivo `SELECTED_OUTPUT` pueden graficarse y exportarse como imagen. La versión que usaremos tiene una interfaz gráfica sencilla que permite modelar fácilmente, cambiar entre distintas bases de datos y generar gráficos tipo x-y.

---

## I. Primeros pasos

El comando `SOLUTION` indica que las especies químicas listadas se disolverán en **1 kg de agua**. Las palabras clave siempre deben escribirse en **mayúsculas**.

El primer archivo de entrada ilustra cómo se utiliza este comando (**archivo de entrada 1**). Puedes copiar directamente el texto del manual y pegarlo en la pestaña de entrada de PHREEQC o escribirlo manualmente.

Todos los archivos de entrada a los que se hace referencia estarán en *cursiva* y listados en el apéndice del manual. En la ventana de PHREEQC también encontrarás un **menú lateral con las palabras clave** disponibles. Si haces doble clic en una de ellas, se insertará automáticamente en el área de entrada.

In [ ]:
import subprocess

# Step 1: Create the PHREEQC input file
input_file_content = """
SOLUTION 1 Mt. Edna Solciechiata
   units      mg/l
   pH         6.90
   pe         -0.2
   density    1.00
   temp       17.2
   Al         0.01
   C(4)       1180 as HCO3
   Ba         0.0296
   Ca         28.1
   Cd         0.00026
   Cl         238 charge
   Cu         0.0018
   Fe         0.150
   K          30.1
   Li         0.180
   Mg         214
   Mn         0.0277
   Na         250
   Pb         0.017
   S(6)       128
   Si         79.4 as SiO2
   Sr         0.673
   Zn         0.0604
END
"""

# ========================== NO TOCAR ESTA PARTE ================================================
# Save the input file
input_file_name = "phreeqc_input_file.pqi"
with open(input_file_name, "w") as file:
    file.write(input_file_content)
print(f"PHREEQC input file '{input_file_name}' created successfully.")

# Step 2: Run PHREEQC using subprocess
output_file_name = "phreeqc_output.txt"
database_file = "/Users/cpinilla/software/phreeqc-3.7.3-15968/database/phreeqc.dat"  # Update the path if necessary
phreeqc_executable = "/Users/cpinilla/software/phreeqc-3.7.3-15968/src/phreeqc"  # Use "phreeqc.exe" on Windows, or the full path to the executable

# Run PHREEQC
try:
    subprocess.run([phreeqc_executable, input_file_name, output_file_name, database_file], check=True)
    print(f"PHREEQC run completed. Output saved in '{output_file_name}'.")
except subprocess.CalledProcessError as e:
    print(f"PHREEQC execution failed: {e}")
    
# Display the contents of the output file, ignoring problematic characters
try:
    with open(output_file_name, "r", encoding="utf-8", errors="ignore") as output_file:
        output_content = output_file.read()
    print("PHREEQC Output:\n")
    print(output_content)
except FileNotFoundError:
    print(f"Output file '{output_file_name}' not found.")    

# II. Modelo Químico del Agua de Mar

Como primer modelo químico, se calcula la distribución de especies en el agua de mar superficial, un problema inicialmente abordado por **Garrels y Thompson (1962)** (ver también Thompson, 1992). Basamos nuestro cálculo en la **composición elemental mayoritaria del agua de mar** (ver Tabla 1.1), determinada mediante análisis químico.

Para establecer el pH, se asume equilibrio con el CO₂ atmosférico (ver Tabla 1.2). Dado que el programa determina la actividad del HCO₃⁻ y de otras especies acuosas, fijar la presión parcial de CO₂ determina el pH según la siguiente reacción:

$$
\mathrm{H}^{+} + \mathrm{HCO}_3^{-} \rightleftharpoons \mathrm{CO}_2(\mathrm{g}) + \mathrm{H}_2\mathrm{O}
$$

De forma similar, el estado de oxidación se define especificando la presión parcial de oxígeno en la atmósfera, lo cual afecta el equilibrio:

$$
\mathrm{O}_2(\mathrm{aq}) \rightleftharpoons \mathrm{O}_2(\mathrm{g}) 
$$

---

### Tabla 1.1 – Composición mayoritaria del agua de mar (Drever, 1988)

| Especie        | Concentración (mg/kg) |
|----------------|------------------------|
| Cl⁻            | 19 350                 |
| Na⁺            | 10 760                 |
| SO₄²⁻          | 2 710                  |
| Mg²⁺           | 1 290                  |
| Ca²⁺           | 411                    |
| K⁺             | 399                    |
| HCO₃⁻          | 142                    |
| SiO₂ (aq)      | 0.5 – 10               |
| O₂ (aq)        | 0.1 – 6                |

---

Estos valores se utilizan como base para definir la solución en los archivos de entrada de **PHREEQC**, utilizando la palabra clave `SOLUTION` y expresando las concentraciones en unidades de `mg/kgw`, equivalentes a `mg/L` en soluciones diluidas.

### Tabla 1.2 – Presiones parciales de algunos gases en la atmósfera (Hem, 1985)

| Gas     | Presión (atm)             |
|---------|---------------------------|
| N₂      | 0.78                      |
| O₂      | 0.21                      |
| H₂O     | 0.001 – 0.23              |
| CO₂     | 0.0003                    |
| CH₄     | 1.5 × 10⁻⁶               |
| CO      | (0.06 – 1) × 10⁻⁶        |
| SO₂     | 1 × 10⁻⁶                 |
| N₂O     | 5 × 10⁻⁷                 |
| H₂      | ~ 5 × 10⁻⁷               |
| NO₂     | (0.05 – 2) × 10⁻⁸        |



## Modelo químico del agua de mar en equilibrio con la atmósfera

En este ejemplo se define un modelo químico simplificado del agua de mar superficial, utilizando composiciones promedio de los principales iones disueltos en el océano (ver tablas anteriores). Este modelo tiene como objetivo calcular la **especiación química** y los **índices de saturación** de minerales potencialmente presentes en el sistema.

El modelo asume que el agua de mar está en equilibrio con la atmósfera en dos aspectos clave:

- **Equilibrio con el CO₂(g)**: esto determina el pH del sistema, según la reacción:
  
- **Equilibrio redox con el O₂(g)**: se controla la disponibilidad de oxígeno disuelto, afectando el estado de oxidación de especies como hierro, manganeso, azufre, entre otros:

Se usa el módulo `EQUILIBRIUM_PHASES` para fijar la presión parcial de estos gases, permitiendo que PHREEQC ajuste las concentraciones de especies acuosas y determine el pH y potencial redox (`pe`) correspondientes al equilibrio con la atmósfera.

Los datos están expresados en `mg/kgw`, que equivale aproximadamente a `mg/L` para soluciones diluidas. El objetivo principal de esta simulación es observar cómo se distribuyen las especies químicas en solución y cómo el equilibrio con la atmósfera afecta el pH y las reacciones químicas en el sistema marino.


In [ ]:
import subprocess

# Step 1: Create the PHREEQC input file
input_file_content = """

SOLUTION 1 Seawater-like composition
    units     mg/kgw
    temp      25
    pH        7.0    charge
    pe        4.0
    redox     pe
    Cl        19350
    Ca        411
    Mg        1290
    Na        10760
    K         399
    S(6)      2710   as SO4
    C(4)      142    as HCO3
    Si        6      as SiO2
    -water    1 # kg of water

EQUILIBRIUM_PHASES 1
    CO2(g)   -3.5   10
    O2(g)    -0.7   10

SELECTED_OUTPUT
    -file seawater_output.txt
    -high_precision true
    -molalities true

END
"""

# ========================== NO TOCAR ESTA PARTE ================================================

# Save the input file
input_file_name = "sea_water_speciation.pqi"
with open(input_file_name, "w") as file:
    file.write(input_file_content)
print(f"PHREEQC input file '{input_file_name}' created successfully.")

# Run PHREEQC
try:
    subprocess.run([phreeqc_executable, input_file_name, output_file_name, database_file], check=True)
    print(f"PHREEQC run completed. Output saved in '{output_file_name}'.")
except subprocess.CalledProcessError as e:
    print(f"PHREEQC execution failed: {e}")
    
# Display the contents of the output file, ignoring problematic characters
try:
    with open(output_file_name, "r", encoding="utf-8", errors="ignore") as output_file:
        output_content = output_file.read()
    print("PHREEQC Output:\n")
    print(output_content)
except FileNotFoundError:
    print(f"Output file '{output_file_name}' not found.")    


Compare las concentraciones de algunas de estas especies, incluyendo la cantidad de oxigeno disuelto calculada con lo que se conoce para la composición tipica del agua de mar.

## Problema 1

## 🌍 Efecto del Aumento de CO₂ Atmosférico sobre el pH del Océano

### Objetivo
Simular cómo cambia el pH del agua de mar a medida que aumenta la concentración de dióxido de carbono (CO₂) en la atmósfera, un fenómeno directamente asociado al **calentamiento global** y la **acidificación oceánica**.

### Contexto
La concentración actual de CO₂ en la atmósfera es de aproximadamente **420 ppm**, lo que equivale a una presión parcial de alrededor de **log P(CO₂) = -3.5** en atmósferas.

Se espera que, si las emisiones continúan, las concentraciones de CO₂ puedan aumentar hasta:
- **560 ppm (~doble preindustrial): log P(CO₂) ≈ -3.25**
- **840 ppm (escenario extremo): log P(CO₂) ≈ -3.0**

El aumento de CO₂ provoca un descenso del pH oceánico debido a su disolución y formación de ácido carbónico:

$$
\mathrm{CO_2(g)} \rightleftharpoons \mathrm{CO_2(aq)} + \mathrm{H_2O} \rightleftharpoons \mathrm{H_2CO_3} \rightleftharpoons \mathrm{H^+} + \mathrm{HCO_3^-}
$$

Este ejercicio permite explorar este efecto en equilibrio, utilizando una solución marina simplificada.

### Instrucciones
Simula tres escenarios con diferente presión parcial de CO₂:
1. **Condición actual:** log P(CO₂) = -3.5 (≈ 420 ppm)
2. **Escenario de emisiones moderadas:** log P(CO₂) = -3.25 (≈ 560 ppm)
3. **Escenario extremo:** log P(CO₂) = -3.0 (≈ 840 ppm)

Observa los cambios en:
- **pH**
- **Distribución de especies de carbono (CO₂, HCO₃⁻, CO₃²⁻)**
- **Índice de saturación de calcita o aragonita**
